In [1]:
!pip install -U langchain langchain-core langchain-community \
    pydantic duckduckgo-search langchain-experimental \
    langchain-ollama


   ---------------------------------------- 0.0/571.8 kB ? eta -:--:--
   ---------------------------------------- 571.8/571.8 kB 6.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 10.8 MB/s  0:00:00
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   -------------- ------------------------- 2.4/6.4 MB 11.1 MB/s eta 0:00:01
   ----------------------------- ---------- 4.7/6.4 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 6.4/6.4 MB 10.7 MB/s  0:00:00

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.46.4

    Uninstalling pydantic_core-2.46.4:

      Successfully uninstalled pydantic_core-2.46.4

   ---------------------------------------- 0/7 [pydantic-core]
  Attempting uninstall: pydantic
   ---------------------------------------- 0/7 [pydantic-core]
   ----------- ---------------------------- 2/7 [pydantic]
    

## Built-in Tool - DuckDuckGo Search : 

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

Get the latest news today, breaking news, and top news headlines updates from India and around the world. Stay updated on politics, business, sports, entertainment, and more with The Indian Express. Research and investigations point to OpenAI’s agents using multiple internet platforms in order to carry out their own communications — without company authorisation ... AI use is already widespread among 15-year-olds. Across OECD countries, only 14% of students reported never or almost never using AI chatbots for the schoolwork purposes examined ... India’s skilling effort is largely supply-led: training happens where the people are, not where the jobs are 2 days ago - In the state capital, 21-year-old Congress candidate Faiz Hasan won ward 71 of the Jaipur Municipal Corporation while Suman Bhati won from ward 72. From BJP, Neha Sharma, 21, has won from Alwar’s Thanagazi municipality ... World champion D Gukesh will be on board 4 for India at the Chess Olympiad, with Praggnanandhaa, Arjun 

In [5]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


## Built-in Tool - Shell Tool : 

In [7]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('dir')

print(results)

Executing command:
 dir
 Volume in drive C is OS
 Volume Serial Number is B889-BA69

 Directory of C:\Users\ANUBHAV\Program_Folders\AI\GenAI\Agents

16-09-2026  20:19    <DIR>          .
16-09-2026  20:10    <DIR>          ..
16-09-2026  20:14    <DIR>          .ipynb_checkpoints
16-09-2026  20:12    <DIR>          Datasets
16-09-2026  20:19            34,584 tools-in-langchain.ipynb
               1 File(s)         34,584 bytes
               4 Dir(s)  362,071,162,880 bytes free



## Custom tool : 

In [8]:
from langchain_core.tools import tool

In [9]:
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [10]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [11]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [12]:
result = multiply.invoke({"a":3, "b":5})

In [13]:
print(result)

15


In [14]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [15]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## Method 2 - Using StructuredTool : 




In [17]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field


In [19]:
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

In [20]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [21]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [22]:
result = multiply_tool.invoke({'a':3, 'b':3})

In [23]:
result

9

In [24]:
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class : 

In [25]:
from langchain_core.tools import BaseTool
from typing import Type

In [27]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to add")
    b: int = Field(description="The second number to add")

In [28]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [29]:
multiply_tool = MultiplyTool()

In [30]:
result = multiply_tool.invoke({'a':3, 'b':3})
result

9

In [31]:
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


## Toolkit : 

In [32]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [33]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [34]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
